# QAOA-GPT inference demo

In [1]:
from model_interface import QAOA_GPT

In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import random
from tqdm import tqdm
from collections import defaultdict, Counter
import json
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


## Loading a model

In [3]:
qaoa_gpt_n10_obj = QAOA_GPT(
    model_ckpt='nanoGPT/models/n10w_qaoa_mixer/ckpt_16000_gemb__ar_0_96584__er_0_0.pt',
    data_dir='nanoGPT/models/n10w_qaoa_mixer/data', # to take meta.pkl file 
    config_file='nanoGPT/models/n10w_qaoa_mixer/data/train_adapt_gpt_config.py',
    temp_folder='temp_data',
    device='cuda'
)

Loading config from: nanoGPT/models/n10w_qaoa_mixer/data/train_adapt_gpt_config.py
Initiating nanoGPT model with padding support
number of parameters: 11.60M


## Generating random graphs

In [4]:
def add_weights_to_nx_graph(nx_graph):
    for u, v in nx_graph.edges():
        cur_weight = round(random.uniform(0, 1), 2)
        while cur_weight == 0:
            cur_weight = round(random.uniform(0, 1), 2)
        nx_graph[u][v]['weight'] = cur_weight
    return nx_graph

In [5]:
tqdm.pandas()

Modify this nodes here to match the model before run

In [6]:
n_graphs = 50
n_nodes = 10

In [7]:
graphs_edgelist_list_dict = dict()

er_graphs_edgelist_list_dict = dict()
for i in range(n_graphs):
    p = random.randrange(6,9) / 10
    cur_graph = nx.erdos_renyi_graph(
        n=n_nodes,
        p=p
    )
    
    er_graphs_edgelist_list_dict[f'er_graph_{i}'] = add_weights_to_nx_graph(cur_graph)

graphs_edgelist_list_dict.update(er_graphs_edgelist_list_dict)

In [8]:
graphs_edgelist_list_dict['er_graph_2'].edges(data=True)

EdgeDataView([(0, 1, {'weight': 0.88}), (0, 2, {'weight': 0.79}), (0, 3, {'weight': 0.33}), (0, 4, {'weight': 0.16}), (0, 5, {'weight': 0.44}), (0, 6, {'weight': 0.84}), (0, 7, {'weight': 0.82}), (0, 9, {'weight': 0.46}), (1, 2, {'weight': 0.06}), (1, 3, {'weight': 0.41}), (1, 4, {'weight': 0.39}), (1, 5, {'weight': 0.86}), (1, 6, {'weight': 0.13}), (1, 7, {'weight': 0.2}), (1, 9, {'weight': 0.15}), (2, 3, {'weight': 0.13}), (2, 4, {'weight': 0.8}), (2, 5, {'weight': 0.14}), (2, 6, {'weight': 0.93}), (2, 7, {'weight': 0.51}), (2, 8, {'weight': 0.45}), (2, 9, {'weight': 0.69}), (3, 6, {'weight': 0.16}), (3, 7, {'weight': 0.15}), (3, 8, {'weight': 0.33}), (3, 9, {'weight': 0.89}), (4, 5, {'weight': 0.43}), (4, 6, {'weight': 0.23}), (4, 8, {'weight': 0.31}), (4, 9, {'weight': 0.52}), (5, 6, {'weight': 0.47}), (5, 7, {'weight': 0.38}), (5, 8, {'weight': 0.95}), (5, 9, {'weight': 0.79}), (6, 7, {'weight': 0.26}), (6, 8, {'weight': 0.01}), (7, 8, {'weight': 0.94}), (8, 9, {'weight': 0.41})])

## Generate circuits with QAOA-GPT model

In [9]:
qaoa_gpt_circ_df = qaoa_gpt_n10_obj.generate_circ_from_nx(
    graphs_edgelist_list_dict,
    # calculate_classic_maxcut=True, # to create col enery_gurobi. Default:True
    n_samples_per_batch=50, # max number of distinct graphs in a batch
    num_samples=5, # number of samples to draw
    max_new_tokens=150, # number of tokens generated in each sample
    temperature=0.1, # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
    top_k=200, # retain only the top_k most likely tokens, clamp others to have 0 probability
)

Preparing graphs...:   0%|          | 0/50 [00:00<?, ?it/s]

Restricted license - for non-production use only - expires 2027-11-29


Preparing graphs...: 100%|██████████| 50/50 [00:00<00:00, 170.46it/s]


Performing FEATHER embedding


100%|██████████| 50/50 [00:00<00:00, 1683.24it/s]
Inference. Current batch: n_edges: 20, n_graphs: 1: 100%|██████████| 18/18 [00:13<00:00,  1.37it/s]


In [10]:
qaoa_gpt_n10_obj.graphs_nx_df[:1]

,graph_id,elist,n_nodes,energy_gurobi,token_seq_round_d2,edgelist_list_len,approx_ratio,label,edgelist_json,has_emb
0,er_graph_0,"[(1, 2, 0.04), (1, 3, 0.5), (1, 4, 0.3), (1, 6, 0.56), (1, 8, 0.38), (1, 9, 0.36), (1, 10, 0.71), (2, 3, 0.88), (2, 4, 0.98), (2, 5, 0.42), (2, 6, 0.21), (2, 7, 0.12), (2, 8, 0.54), (2, 9, 0.46), (2, 10, 0.38), (3, 4, 0.42), (3, 5, 0.78), (3, 7, 0.99), (3, 8, 0.72), (3, 9, 0.06), (4, 7, 0.77), (4, 8, 0.07), (4, 9, 0.26), (4, 10, 0.21), (5, 6, 0.05), (5, 7, 0.01), (5, 9, 0.21), (5, 10, 0.01), (6, 7, 0.62), (6, 8, 0.68), (6, 10, 0.75), (7, 8, 0.96), (8, 9, 0.97), (8, 10, 0.33), (9, 10, 0.7)]",10,-11.77,"[bos, (1, 2), 0.04, (1, 3), 0.5, (1, 4), 0.3, (1, 6), 0.56, (1, 8), 0.38, (1, 9), 0.36, (1, 10), 0.71, (2, 3), 0.88, (2, 4), 0.98, (2, 5), 0.42, (2, 6), 0.21, (2, 7), 0.12, (2, 8), 0.54, (2, 9), 0.46, (2, 10), 0.38, (3, 4), 0.42, (3, 5), 0.78, (3, 7), 0.99, (3, 8), 0.72, (3, 9), 0.06, (4, 7), 0.77, (4, 8), 0.07, (4, 9), 0.26, (4, 10), 0.21, (5, 6), 0.05, (5, 7), 0.01, (5, 9), 0.21, (5, 10), 0.01, (6, 7), 0.62, (6, 8), 0.68, (6, 10), 0.75, (7, 8), 0.96, (8, 9), 0.97, (8, 10), 0.33, (9, 10), 0.7, end_of_graph]",35,None,test_interactive,"[[1, 2, 0.04], [1, 3, 0.5], [1, 4, 0.3], [1, 6, 0.56], [1, 8, 0.38], [1, 9, 0.36], [1, 10, 0.71], [2, 3, 0.88], [2, 4, 0.98], [2, 5, 0.42], [2, 6, 0.21], [2, 7, 0.12], [2, 8, 0.54], [2, 9, 0.46], [2, 10, 0.38], [3, 4, 0.42], [3, 5, 0.78], [3, 7, 0.99], [3, 8, 0.72], [3, 9, 0.06], [4, 7, 0.77], [4, 8, 0.07], [4, 9, 0.26], [4, 10, 0.21], [5, 6, 0.05], [5, 7, 0.01], [5, 9, 0.21], [5, 10, 0.01], [6, 7, 0.62], [6, 8, 0.68], [6, 10, 0.75], [7, 8, 0.96], [8, 9, 0.97], [8, 10, 0.33], [9, 10, 0.7]]",True


In [11]:
qaoa_gpt_n10_obj.feather_par_emb[0][:10]

array([ 0.47,  0.46,  0.42,  0.35,  0.26,  0.16,  0.05, -0.06, -0.17,
       -0.27])

In [12]:
# qaoa_gpt_n10_obj.meta

In [13]:
# qaoa_gpt_n10_obj.emb_graph_id_to_idx_dict

In [14]:
sample_gr = graphs_edgelist_list_dict['er_graph_0'].edges(data=True)
sample_gr

EdgeDataView([(0, 1, {'weight': 0.04}), (0, 2, {'weight': 0.5}), (0, 3, {'weight': 0.3}), (0, 5, {'weight': 0.56}), (0, 7, {'weight': 0.38}), (0, 8, {'weight': 0.36}), (0, 9, {'weight': 0.71}), (1, 2, {'weight': 0.88}), (1, 3, {'weight': 0.98}), (1, 4, {'weight': 0.42}), (1, 5, {'weight': 0.21}), (1, 6, {'weight': 0.12}), (1, 7, {'weight': 0.54}), (1, 8, {'weight': 0.46}), (1, 9, {'weight': 0.38}), (2, 3, {'weight': 0.42}), (2, 4, {'weight': 0.78}), (2, 6, {'weight': 0.99}), (2, 7, {'weight': 0.72}), (2, 8, {'weight': 0.06}), (3, 6, {'weight': 0.77}), (3, 7, {'weight': 0.07}), (3, 8, {'weight': 0.26}), (3, 9, {'weight': 0.21}), (4, 5, {'weight': 0.05}), (4, 6, {'weight': 0.01}), (4, 8, {'weight': 0.21}), (4, 9, {'weight': 0.01}), (5, 6, {'weight': 0.62}), (5, 7, {'weight': 0.68}), (5, 9, {'weight': 0.75}), (6, 7, {'weight': 0.96}), (7, 8, {'weight': 0.97}), (7, 9, {'weight': 0.33}), (8, 9, {'weight': 0.7})])

In [15]:
len(graphs_edgelist_list_dict['er_graph_0'].edges(data=True))

35

The graph after prediction is shifted by 1 unit. For example, in NetworkX the edge is (0, 2, 0.36), but in this DataFrame it becomes (1, 3, 0.36)

In [16]:
qaoa_gpt_circ_df[:1]

,graph,n_edges,q_circuits,adapt_circuit,adapt_full_ar,graph_prefix,energy_gurobi,label,graph_w_jl,graph_weight_norm
0,"[(1, 3), 0.56, (1, 5), 0.95, (1, 6), 0.51, (1, 8), 0.64, (1, 9), 0.83, (2, 3), 0.95, (2, 6), 0.75, (2, 7), 0.27, (2, 10), 0.33, (3, 5), 0.14, (3, 6), 0.92, (3, 8), 0.53, (3, 9), 0.3, (3, 10), 0.86, (4, 5), 0.96, (4, 6), 0.22, (4, 8), 0.96, (4, 9), 0.44, (4, 10), 0.01, (5, 6), 0.46, (5, 7), 0.31, (5, 10), 0.03, (6, 7), 0.72, (7, 8), 0.36, (7, 9), 0.03, (7, 10), 0.16, (8, 9), 0.04]",27,"[[new_layer_p, 1, -0.56, 0.32, new_layer_p, 1, -0.44, 0.65, new_layer_p, 1, -0.38, 0.71, new_layer_p, 1, -0.35, 0.75, new_layer_p, 1, -0.31, 0.81, new_layer_p, 1, -0.25, 0.91, new_layer_p, 1, -0.12, 1.07], [new_layer_p, 1, -0.57, 0.32, new_layer_p, 1, -0.46, 0.65, new_layer_p, 1, -0.39, 0.72, new_layer_p, 1, -0.35, 0.76, new_layer_p, 1, -0.31, 0.81, new_layer_p, 1, -0.27, 0.87, new_layer_p, 1, -0.2, 0.97, new_layer_p, 1, -0.1, 1.12], [new_layer_p, 1, -0.57, 0.33, new_layer_p, 1, -0.46, 0.66, new_layer_p, 1, -0.39, 0.72, new_layer_p, 1, -0.33, 0.77, new_layer_p, 1, -0.25, 0.84, new_layer_p, 1, -0.12, 0.92], [new_layer_p, 1, -0.57, 0.31, new_layer_p, 1, -0.45, 0.63, new_layer_p, 1, -0.38, 0.7, new_layer_p, 1, -0.34, 0.75, new_layer_p, 1, -0.29, 0.81, new_layer_p, 1, -0.22, 0.91, new_layer_p, 1, -0.15, 1, new_layer_p, 1, -0.08, 1.08], [new_layer_p, 1, -0.57, 0.32, new_layer_p, 1, -0.46, 0.65, new_layer_p, 1, -0.39, 0.72, new_layer_p, 1, -0.35, 0.78, new_layer_p, 1, -0.31, 0.83, new_layer_p, 1, -0.26, 0.9, new_layer_p, 1, -0.19, 1, new_layer_p, 1, -0.1, 1.15]]",[],None,er_graph_13,-11.07,test_interactive,"[[1, 3, 0.56], [1, 5, 0.95], [1, 6, 0.51], [1, 8, 0.64], [1, 9, 0.83], [2, 3, 0.95], [2, 6, 0.75], [2, 7, 0.27], [2, 10, 0.33], [3, 5, 0.14], [3, 6, 0.92], [3, 8, 0.53], [3, 9, 0.3], [3, 10, 0.86], [4, 5, 0.96], [4, 6, 0.22], [4, 8, 0.96], [4, 9, 0.44], [4, 10, 0.01], [5, 6, 0.46], [5, 7, 0.31], [5, 10, 0.03], [6, 7, 0.72], [7, 8, 0.36], [7, 9, 0.03], [7, 10, 0.16], [8, 9, 0.04]]",1.0


In [17]:
qaoa_gpt_circ_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   graph              50 non-null     object 
 1   n_edges            50 non-null     int64  
 2   q_circuits         50 non-null     object 
 3   adapt_circuit      50 non-null     object 
 4   adapt_full_ar      0 non-null      object 
 5   graph_prefix       50 non-null     object 
 6   energy_gurobi      50 non-null     float64
 7   label              50 non-null     object 
 8   graph_w_jl         50 non-null     object 
 9   graph_weight_norm  50 non-null     float64
dtypes: float64(2), int64(1), object(7)
memory usage: 4.0+ KB


## Evaluate circuits

In [18]:
qaoa_gpt_circ_eval_df = qaoa_gpt_n10_obj.eval_circ_df_jl(qaoa_gpt_circ_df)

In [19]:
sample_gr

EdgeDataView([(0, 1, {'weight': 0.04}), (0, 2, {'weight': 0.5}), (0, 3, {'weight': 0.3}), (0, 5, {'weight': 0.56}), (0, 7, {'weight': 0.38}), (0, 8, {'weight': 0.36}), (0, 9, {'weight': 0.71}), (1, 2, {'weight': 0.88}), (1, 3, {'weight': 0.98}), (1, 4, {'weight': 0.42}), (1, 5, {'weight': 0.21}), (1, 6, {'weight': 0.12}), (1, 7, {'weight': 0.54}), (1, 8, {'weight': 0.46}), (1, 9, {'weight': 0.38}), (2, 3, {'weight': 0.42}), (2, 4, {'weight': 0.78}), (2, 6, {'weight': 0.99}), (2, 7, {'weight': 0.72}), (2, 8, {'weight': 0.06}), (3, 6, {'weight': 0.77}), (3, 7, {'weight': 0.07}), (3, 8, {'weight': 0.26}), (3, 9, {'weight': 0.21}), (4, 5, {'weight': 0.05}), (4, 6, {'weight': 0.01}), (4, 8, {'weight': 0.21}), (4, 9, {'weight': 0.01}), (5, 6, {'weight': 0.62}), (5, 7, {'weight': 0.68}), (5, 9, {'weight': 0.75}), (6, 7, {'weight': 0.96}), (7, 8, {'weight': 0.97}), (7, 9, {'weight': 0.33}), (8, 9, {'weight': 0.7})])

The eval df add 1 more col is adapt_gpt_energies

Each graph is generate into 5 other graphs, that why we see list of 5 q_circuits and adapt_gpt_energies



In [20]:
qaoa_gpt_circ_eval_df[:1]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_13,"[[1, 3], 0.56, [1, 5], 0.9500000000000001, [1, 6], 0.51, [1, 8], 0.64, [1, 9], 0.8300000000000001, [2, 3], 0.9500000000000001, [2, 6], 0.75, [2, 7], 0.27, [2, 10], 0.33, [3, 5], 0.14, [3, 6], 0.92, [3, 8], 0.53, [3, 9], 0.30000000000000004, [3, 10], 0.86, [4, 5], 0.96, [4, 6], 0.22, [4, 8], 0.96, [4, 9], 0.44, [4, 10], 0.01, [5, 6], 0.46, [5, 7], 0.31, [5, 10], 0.03, [6, 7], 0.72, [7, 8], 0.36, [7, 9], 0.03, [7, 10], 0.16, [8, 9], 0.04]",27,"[[new_layer_p, 1, -0.56, 0.32, new_layer_p, 1, -0.44, 0.65, new_layer_p, 1, -0.38, 0.71, new_layer_p, 1, -0.35000000000000003, 0.75, new_layer_p, 1, -0.31, 0.81, new_layer_p, 1, -0.25, 0.91, new_layer_p, 1, -0.12, 1.07], [new_layer_p, 1, -0.5700000000000001, 0.32, new_layer_p, 1, -0.46, 0.65, new_layer_p, 1, -0.39, 0.72, new_layer_p, 1, -0.35000000000000003, 0.76, new_layer_p, 1, -0.31, 0.81, new_layer_p, 1, -0.27, 0.87, new_layer_p, 1, -0.2, 0.97, new_layer_p, 1, -0.1, 1.12], [new_layer_p, 1, -0.5700000000000001, 0.33, new_layer_p, 1, -0.46, 0.66, new_layer_p, 1, -0.39, 0.72, new_layer_p, 1, -0.33, 0.77, new_layer_p, 1, -0.25, 0.84, new_layer_p, 1, -0.12, 0.92], [new_layer_p, 1, -0.5700000000000001, 0.31, new_layer_p, 1, -0.45, 0.63, new_layer_p, 1, -0.38, 0.7000000000000001, new_layer_p, 1, -0.34, 0.75, new_layer_p, 1, -0.29, 0.81, new_layer_p, 1, -0.22, 0.91, new_layer_p, 1, -0.15, 1, new_layer_p, 1, -0.08, 1.08], [new_layer_p, 1, -0.5700000000000001, 0.32, new_layer_p, 1, -0.46, 0.65, new_layer_p, 1, -0.39, 0.72, new_layer_p, 1, -0.35000000000000003, 0.78, new_layer_p, 1, -0.31, 0.8300000000000001, new_layer_p, 1, -0.26, 0.9, new_layer_p, 1, -0.19, 1, new_layer_p, 1, -0.1, 1.15]]","[-10.718304582390406, -10.796055916854753, -10.564390489730123, -10.73982515263429, -10.776929719565079]",-11.07


In [21]:
qaoa_gpt_circ_eval_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   graph_prefix        50 non-null     object 
 1   graph               50 non-null     object 
 2   n_edges             50 non-null     int64  
 3   q_circuits          50 non-null     object 
 4   adapt_gpt_energies  50 non-null     object 
 5   energy_gurobi       50 non-null     float64
dtypes: float64(1), int64(1), object(4)
memory usage: 2.5+ KB


In [22]:
# 3 extract first rows into 5 rows for 5 q_circuits and adapt_gpt_energies
qaoa_gpt_circ_eval_expl_df = qaoa_gpt_circ_eval_df.explode(['adapt_gpt_energies', 'q_circuits']) 

In [23]:
qaoa_gpt_circ_eval_expl_df[:5]

,graph_prefix,graph,n_edges,q_circuits,adapt_gpt_energies,energy_gurobi
0,er_graph_13,"[[1, 3], 0.56, [1, 5], 0.9500000000000001, [1, 6], 0.51, [1, 8], 0.64, [1, 9], 0.8300000000000001, [2, 3], 0.9500000000000001, [2, 6], 0.75, [2, 7], 0.27, [2, 10], 0.33, [3, 5], 0.14, [3, 6], 0.92, [3, 8], 0.53, [3, 9], 0.30000000000000004, [3, 10], 0.86, [4, 5], 0.96, [4, 6], 0.22, [4, 8], 0.96, [4, 9], 0.44, [4, 10], 0.01, [5, 6], 0.46, [5, 7], 0.31, [5, 10], 0.03, [6, 7], 0.72, [7, 8], 0.36, [7, 9], 0.03, [7, 10], 0.16, [8, 9], 0.04]",27,"[new_layer_p, 1, -0.56, 0.32, new_layer_p, 1, -0.44, 0.65, new_layer_p, 1, -0.38, 0.71, new_layer_p, 1, -0.35000000000000003, 0.75, new_layer_p, 1, -0.31, 0.81, new_layer_p, 1, -0.25, 0.91, new_layer_p, 1, -0.12, 1.07]",-10.718305,-11.07
0,er_graph_13,"[[1, 3], 0.56, [1, 5], 0.9500000000000001, [1, 6], 0.51, [1, 8], 0.64, [1, 9], 0.8300000000000001, [2, 3], 0.9500000000000001, [2, 6], 0.75, [2, 7], 0.27, [2, 10], 0.33, [3, 5], 0.14, [3, 6], 0.92, [3, 8], 0.53, [3, 9], 0.30000000000000004, [3, 10], 0.86, [4, 5], 0.96, [4, 6], 0.22, [4, 8], 0.96, [4, 9], 0.44, [4, 10], 0.01, [5, 6], 0.46, [5, 7], 0.31, [5, 10], 0.03, [6, 7], 0.72, [7, 8], 0.36, [7, 9], 0.03, [7, 10], 0.16, [8, 9], 0.04]",27,"[new_layer_p, 1, -0.5700000000000001, 0.32, new_layer_p, 1, -0.46, 0.65, new_layer_p, 1, -0.39, 0.72, new_layer_p, 1, -0.35000000000000003, 0.76, new_layer_p, 1, -0.31, 0.81, new_layer_p, 1, -0.27, 0.87, new_layer_p, 1, -0.2, 0.97, new_layer_p, 1, -0.1, 1.12]",-10.796056,-11.07
0,er_graph_13,"[[1, 3], 0.56, [1, 5], 0.9500000000000001, [1, 6], 0.51, [1, 8], 0.64, [1, 9], 0.8300000000000001, [2, 3], 0.9500000000000001, [2, 6], 0.75, [2, 7], 0.27, [2, 10], 0.33, [3, 5], 0.14, [3, 6], 0.92, [3, 8], 0.53, [3, 9], 0.30000000000000004, [3, 10], 0.86, [4, 5], 0.96, [4, 6], 0.22, [4, 8], 0.96, [4, 9], 0.44, [4, 10], 0.01, [5, 6], 0.46, [5, 7], 0.31, [5, 10], 0.03, [6, 7], 0.72, [7, 8], 0.36, [7, 9], 0.03, [7, 10], 0.16, [8, 9], 0.04]",27,"[new_layer_p, 1, -0.5700000000000001, 0.33, new_layer_p, 1, -0.46, 0.66, new_layer_p, 1, -0.39, 0.72, new_layer_p, 1, -0.33, 0.77, new_layer_p, 1, -0.25, 0.84, new_layer_p, 1, -0.12, 0.92]",-10.56439,-11.07
0,er_graph_13,"[[1, 3], 0.56, [1, 5], 0.9500000000000001, [1, 6], 0.51, [1, 8], 0.64, [1, 9], 0.8300000000000001, [2, 3], 0.9500000000000001, [2, 6], 0.75, [2, 7], 0.27, [2, 10], 0.33, [3, 5], 0.14, [3, 6], 0.92, [3, 8], 0.53, [3, 9], 0.30000000000000004, [3, 10], 0.86, [4, 5], 0.96, [4, 6], 0.22, [4, 8], 0.96, [4, 9], 0.44, [4, 10], 0.01, [5, 6], 0.46, [5, 7], 0.31, [5, 10], 0.03, [6, 7], 0.72, [7, 8], 0.36, [7, 9], 0.03, [7, 10], 0.16, [8, 9], 0.04]",27,"[new_layer_p, 1, -0.5700000000000001, 0.31, new_layer_p, 1, -0.45, 0.63, new_layer_p, 1, -0.38, 0.7000000000000001, new_layer_p, 1, -0.34, 0.75, new_layer_p, 1, -0.29, 0.81, new_layer_p, 1, -0.22, 0.91, new_layer_p, 1, -0.15, 1, new_layer_p, 1, -0.08, 1.08]",-10.739825,-11.07
0,er_graph_13,"[[1, 3], 0.56, [1, 5], 0.9500000000000001, [1, 6], 0.51, [1, 8], 0.64, [1, 9], 0.8300000000000001, [2, 3], 0.9500000000000001, [2, 6], 0.75, [2, 7], 0.27, [2, 10], 0.33, [3, 5], 0.14, [3, 6], 0.92, [3, 8], 0.53, [3, 9], 0.30000000000000004, [3, 10], 0.86, [4, 5], 0.96, [4, 6], 0.22, [4, 8], 0.96, [4, 9], 0.44, [4, 10], 0.01, [5, 6], 0.46, [5, 7], 0.31, [5, 10], 0.03, [6, 7], 0.72, [7, 8], 0.36, [7, 9], 0.03, [7, 10], 0.16, [8, 9], 0.04]",27,"[new_layer_p, 1, -0.5700000000000001, 0.32, new_layer_p, 1, -0.46, 0.65, new_layer_p, 1, -0.39, 0.72, new_layer_p, 1, -0.35000000000000003, 0.78, new_layer_p, 1, -0.31, 0.8300000000000001, new_layer_p, 1, -0.26, 0.9, new_layer_p, 1, -0.19, 1, new_layer_p, 1, -0.1, 1.15]",-10.77693,-11.07


This computes how close each QAOA-GPT circuit’s energy is to the optimal solution (AR — approximation ratio).

If the ratio = 1 → perfect solution.

If <1 → circuit energy is above the optimal energy (since MaxCut is a minimization problem in some conventions, sometimes they flip it; conceptually, ratio shows performance).

In [24]:
(qaoa_gpt_circ_eval_expl_df['adapt_gpt_energies'] / qaoa_gpt_circ_eval_expl_df['energy_gurobi']).mean()

np.float64(0.9662990362930617)

In [25]:
# on avg, how many layers are there in the predicted circuits
qaoa_gpt_circ_eval_expl_df['q_circuits'].apply(lambda x: x.count('new_layer_p')).mean()

np.float64(8.556)